# T2I-RL 快速训练 (SiliconFlow 免费 VLM)

**使用国产免费 API 进行 VLM Reward 训练**

## 优势
- **超低成本**: SiliconFlow 提供极低价格的 VLM API
- **国内访问快**: 无需翻墙
- **兼容 OpenAI 格式**: 代码改动最小

## 训练成本估算
- 100 prompts × 4 images × 5 epochs = 2000 次调用
- 约 ¥0.4 总成本 (Qwen2.5-VL-32B)

---

## Step 0: 配置 API Key

**获取 API Key**: https://cloud.siliconflow.cn/

1. 注册账号
2. 进入控制台
3. 创建 API Key

In [ ]:
# ========================================
# 配置 SiliconFlow API Key
# ========================================

SILICONFLOW_API_KEY = "sk-qdbrvnasyqcpqsvekchvkvrsuikujzfwgoltoytnegnlvdbk"  # 替换为你的 API Key

# 选择 VLM 模型
# 推荐: "Qwen/Qwen2.5-VL-32B-Instruct" (高性价比)
# 免费: "THUDM/GLM-4.1V-9B-Thinking" (完全免费但效果略差)
VLM_MODEL = "Qwen/Qwen2.5-VL-32B-Instruct"

# ========================================

import os
os.environ["SILICONFLOW_API_KEY"] = SILICONFLOW_API_KEY
print(f"✓ Using SiliconFlow VLM: {VLM_MODEL}")

## Step 1: 安装依赖

In [ ]:
import time
start = time.time()

# 安装基础依赖
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.49.0 accelerate safetensors
!pip install -q --no-deps git+https://github.com/deepseek-ai/Janus.git
!pip install -q attrdict sentencepiece
!pip install -q open-clip-torch timm einops peft bitsandbytes
!pip install -q tqdm Pillow matplotlib

# OpenAI SDK (用于调用 SiliconFlow API)
!pip install -q openai

print(f"\n✓ Installation complete: {time.time()-start:.1f}s")

## Step 2: 测试 VLM API 连接

In [ ]:
from openai import OpenAI
import base64
from PIL import Image
import io

# 创建测试图像
test_img = Image.new('RGB', (256, 256), color='red')
buffer = io.BytesIO()
test_img.save(buffer, format='PNG')
img_base64 = base64.b64encode(buffer.getvalue()).decode()

# 测试 API
client = OpenAI(
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1"
)

response = client.chat.completions.create(
    model=VLM_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What color is this image? Reply in one word."},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{img_base64}"},
                },
            ],
        }
    ],
    max_tokens=100,
)

print(f"✓ VLM API 测试成功!")
print(f"Response: {response.choices[0].message.content}")

## Step 3: 检查 GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU found! Training will be very slow.")

## Step 4: 克隆项目代码

In [ ]:
import os

if not os.path.exists('T2I-RL-Project'):
    !git clone https://github.com/Inoriany/T2I-RL-Project.git
    print("✓ Repository cloned")
else:
    !cd T2I-RL-Project && git pull
    print("✓ Repository updated")

os.chdir('T2I-RL-Project')
print(f"Working directory: {os.getcwd()}")

## Step 5: 加载 Janus-Pro 模型 (4-bit 量化)

In [ ]:
import sys
sys.path.insert(0, 'src')

from models.generators import JanusProGenerator

print("Loading Janus-Pro-1B with 4-bit quantization...")
generator = JanusProGenerator(
    model_name_or_path="deepseek-ai/Janus-Pro-1B",
    device="cuda",
    load_in_4bit=True  # 节省显存
)
generator.load_model()

# 测试生成
test_prompt = "a red apple on a wooden table"
images = generator.generate(prompts=[test_prompt], num_images_per_prompt=1)

print(f"✓ Generator loaded and tested")
display(images[0])

## Step 6: 创建 VLM Reward Model

In [ ]:
from models.reward_models import VLMRewardModel

# 创建 VLM Reward Model (使用 SiliconFlow API)
reward_model = VLMRewardModel(
    device="cuda",
    use_api=True,
    api_model=VLM_MODEL  # 使用 SiliconFlow 的模型
)
reward_model.load_model()

# 测试 reward 计算
test_images = generator.generate(prompts=[test_prompt], num_images_per_prompt=1)
reward_output = reward_model.compute_reward(
    images=test_images,
    prompts=[test_prompt]
)

print(f"✓ VLM Reward Model ready")
print(f"Test reward: {reward_output.rewards.item():.4f}")
print(f"Details: {reward_output.details}")

## Step 7: 准备训练数据

In [ ]:
import json

# 加载训练 prompts
with open('data/prompts/train_prompts.json', 'r') as f:
    all_prompts = json.load(f)

print(f"Total prompts available: {len(all_prompts)}")

# 快速训练: 只用 50 个 prompts
NUM_PROMPTS = 50
train_prompts = all_prompts[:NUM_PROMPTS]

print(f"Using {NUM_PROMPTS} prompts for quick training")
print(f"\nExample prompts:")
for p in train_prompts[:5]:
    print(f"  - {p['prompt']}")

## Step 8: 开始 GRPO 训练

**预计时间**: ~2-3 小时 (50 prompts, 3 epochs)

In [ ]:
from training.grpo_trainer import GRPOTrainer
from training.base_trainer import TrainingConfig
from tqdm.auto import tqdm
import torch

# 训练配置
config = TrainingConfig(
    num_epochs=3,              # 快速训练: 3 epochs
    batch_size=2,              # 小 batch 节省显存
    learning_rate=1e-5,
    num_images_per_prompt=4,   # GRPO: 每个 prompt 生成 4 张图
    gradient_accumulation_steps=4,
    save_every_n_steps=50,
    output_dir="./outputs/grpo_siliconflow_quick",
    use_lora=True,             # 使用 LoRA 节省显存
    lora_r=8,
    lora_alpha=16,
)

# 创建 trainer
trainer = GRPOTrainer(
    generator=generator,
    reward_model=reward_model,
    config=config
)

print("Starting GRPO training with VLM reward...")
print(f"  Prompts: {NUM_PROMPTS}")
print(f"  Epochs: {config.num_epochs}")
print(f"  Batch size: {config.batch_size}")
print(f"  Images per prompt: {config.num_images_per_prompt}")
print(f"  Total training steps: ~{NUM_PROMPTS * config.num_epochs // config.batch_size}")

In [ ]:
# 运行训练
prompt_texts = [p['prompt'] for p in train_prompts]

training_history = trainer.train(prompt_texts)

print("\n" + "="*50)
print("Training Complete!")
print("="*50)

## Step 9: 保存训练好的模型

In [ ]:
# 保存 checkpoint
trainer.save_checkpoint("./outputs/grpo_siliconflow_quick/final_checkpoint.pt")
print("✓ Model checkpoint saved")

# 保存训练历史
import json
with open("./outputs/grpo_siliconflow_quick/training_history.json", "w") as f:
    json.dump(training_history, f, indent=2)
print("✓ Training history saved")

## Step 10: Before vs After 对比

In [ ]:
import matplotlib.pyplot as plt

# 测试 prompts
test_prompts = [
    "a red cat sitting on a blue chair",
    "two green apples and one yellow banana",
    "a small dog under a large table",
]

# 生成 After 图像 (训练后)
after_images = []
after_rewards = []

for prompt in test_prompts:
    img = generator.generate(prompts=[prompt], num_images_per_prompt=1)[0]
    reward = reward_model.compute_reward([img], [prompt]).rewards.item()
    after_images.append(img)
    after_rewards.append(reward)

# 可视化
fig, axes = plt.subplots(1, len(test_prompts), figsize=(15, 5))

for i, (prompt, img, reward) in enumerate(zip(test_prompts, after_images, after_rewards)):
    axes[i].imshow(img)
    axes[i].set_title(f"Reward: {reward:.3f}\n{prompt[:30]}...", fontsize=10)
    axes[i].axis('off')

plt.suptitle("After Training (VLM Reward)", fontsize=14)
plt.tight_layout()
plt.savefig("./outputs/grpo_siliconflow_quick/after_training.png", dpi=150)
plt.show()

print("\nAverage reward after training:", sum(after_rewards) / len(after_rewards))

## Step 11: 下载结果

运行完成后，下载以下文件用于报告:
- `outputs/grpo_siliconflow_quick/final_checkpoint.pt` - 模型权重
- `outputs/grpo_siliconflow_quick/training_history.json` - 训练曲线
- `outputs/grpo_siliconflow_quick/after_training.png` - 生成图像

In [ ]:
# 打包结果
!zip -r results.zip outputs/grpo_siliconflow_quick/

from google.colab import files
files.download('results.zip')

print("✓ Results downloaded!")

---

## 下一步

1. **运行 Benchmark 评估** - 使用 `notebooks/benchmark_evaluation.ipynb`
2. **生成 Before/After Grid** - 使用 `notebooks/visualization.ipynb`
3. **Error Taxonomy 分析** - 使用 `notebooks/error_analysis.ipynb`